In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

PROJECT_ROOT = Path(r"C:\Users\asus\OneDrive\EV-projects\evcs-projects")
BENCH_DIR    = PROJECT_ROOT / "results" / "benchmarking"
EXCEL_FILE   = BENCH_DIR / "benchmark_with_SLURM.xlsx"

df = pd.read_excel(EXCEL_FILE, sheet_name="benchmark")
print(f"Loaded {len(df)} rows")
df[['Timestamp','Instance','N','T','D_km','seed','DR_best','Exact_incumbent_raw','Gap_%']]

In [ ]:
def get_trace(row):
    sheet = row.get("Trace_sheet") if hasattr(row, "get") else row["Trace_sheet"]
    if pd.isna(sheet) or not sheet:
        return None
    try:
        return pd.read_excel(EXCEL_FILE, sheet_name=str(sheet))
    except Exception:
        return None


def plot_dr_curve(ax, trace, row):
    x    = trace["iteration"].to_numpy()
    best = trace["best_full"].ffill().to_numpy()

    if "proxy_mean" in trace.columns:
        current = trace["proxy_mean"].to_numpy()
        lbl = "DR fluctuating (current)"
    elif "proxy_max" in trace.columns:
        current = trace["proxy_max"].to_numpy()
        lbl = "DR fluctuating (proxy max)"
    else:
        current = best
        lbl = "DR fluctuating"

    ax.plot(x, current, linewidth=1.0, alpha=0.55, color="tab:blue", label=lbl)
    ax.plot(x, best, linewidth=2.5, color="tab:orange", label="DR best-so-far (best_full)")

    exact = row["Exact_incumbent_raw"]
    if pd.notna(exact):
        ax.axhline(y=float(exact), linestyle="--", linewidth=2.0, color="tab:blue",
                   label=f"Exact ({float(exact):.3f})")

    gap     = row["Gap_%"]
    gap_str = f"{gap:.4f}%" if pd.notna(gap) else "N/A"
    policy  = row["Policy"] if "Policy" in row.index else "N/A"
    dr_its  = int(row["DR_iters"]) if "DR_iters" in row.index and pd.notna(row["DR_iters"]) else len(x)

    ax.set_title(
        f"DR vs Exact | N={int(row['N'])}, T={int(row['T'])}, seed={int(row['seed'])} | policy={policy}
"
        f"Gap={gap_str}   DR={best[-1]:.3f}   Exact={float(exact):.3f}   iters={dr_its}",
        fontsize=9
    )
    ax.set_xlabel("iteration", fontsize=9)
    ax.set_ylabel("Score", fontsize=9)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(fontsize=8)
    ax.grid(True, linewidth=0.5)


print("Helper functions ready.")


In [ ]:
# --- collect rows that have trace data embedded in Excel ---
plottable = []
for idx, row in df.iterrows():
    t = get_trace(row)
    if t is not None:
        plottable.append((idx, row, t))
    else:
        print(f"  row {idx}: no trace  (T={int(row['T'])} D={row['D_km']} seed={int(row['seed'])})")

print(f"
{len(plottable)} / {len(df)} rows have trace data.")

if plottable:
    ncols = 2
    nrows = -(-len(plottable) // ncols)   # ceiling division
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 5 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, (idx, row, trace) in enumerate(plottable):
        plot_dr_curve(axes[i], trace, row)

    for j in range(len(plottable), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print("No trace data found. Re-run experiments with the updated run_evcs.py to embed traces.")


In [ ]:
# --- summary table with color-coded gap ---
cols = ['Timestamp','Instance','Policy','N','T','D_km','seed',
        'Exact_incumbent_raw','DR_best','Gap_%','DR_iters','DR_time_s','Exact_time_s']
show = [c for c in cols if c in df.columns]
df[show].style     .format({'Exact_incumbent_raw':'{:.4f}','DR_best':'{:.4f}',
             'Gap_%':'{:.4f}%','DR_time_s':'{:.1f}s','Exact_time_s':'{:.1f}s'})     .background_gradient(subset=['Gap_%'], cmap='RdYlGn_r')
